In [ ]:
# clear all the parameters
dbutils.widgets.removeAll()

In [ ]:
%run "../utils/setup_databricks_session"

In [ ]:
from core.configs import WIDGETS_CONFIG
from core.engine import scoped_merge_engine, unify_source_dataframes
from core.helpers import complete_sample_fact_projection
from config.blueprints.sample_rules import get_sample_blueprint, standardize_sample_schema
from core.databricks_session import create_required_widgets, fetch_widget_values, log, view
from core.data_quality import build_schema_constraints_df, data_quality_engine
from pyspark.sql import functions as F

In [ ]:
# define the required widgets
required_widgets = [
    "execution_mode",
    "bronze_catalog",
    "bronze_schema",
    "silver_catalog",
    "silver_schema",
    "source_cd",
    "sample_1_dim",
    "target_dim",
    "target_sample_fact",
    "target_sample_fact_quarantine",
    "target_currency_exchange_rate_fact",
]

# Create widgets for required parameters
create_required_widgets(dbutils, WIDGETS_CONFIG, required_widgets)

In [ ]:
# Get widget values
params = fetch_widget_values(dbutils, print_validation=False)

# Define other parameters
params.sample_filter1 = ["random1", "random2", "random3"]
params.sample_filter2 = ["random4", "random5"]
log(f"Execution Mode: {params.execution_mode}")

In [ ]:
source_config, target_projection = get_sample_blueprint(
    params.bronze_catalog, params.bronze_schema, params.sample_filter1, params.sample_filter2
)
sample_blueprint = unify_source_dataframes(spark, source_config, target_projection)
view(sample_blueprint.count(), params.execution_mode)
view(sample_blueprint, params.execution_mode)

In [ ]:
standardized_sample = standardize_sample_schema(sample_blueprint)
view(standardized_sample.count(), params.execution_mode)
view(standardized_sample, params.execution_mode)

In [ ]:
currency_exchange_rate_fact_df = spark.table(
    f"{params.silver_catalog}.{params.silver_schema}.{params.target_currency_exchange_rate_fact}"
)
sample_dim_df = spark.table(f"{params.silver_catalog}.{params.silver_schema}.{params.sample_1_dim}")
sample_fact_df = complete_sample_fact_projection(
    standardized_sample, currency_exchange_rate_fact_df, sample_dim_df, params.source_cd
)
view(sample_fact_df.count(), params.execution_mode)
view(sample_fact_df, params.execution_mode)

In [ ]:
columns_df = spark.table(f"{params.silver_catalog}.information_schema.columns")
key_column_usage_df = spark.table(f"{params.silver_catalog}.information_schema.key_column_usage")
table_constraints_df = spark.table(f"{params.silver_catalog}.information_schema.table_constraints")
referential_constraints_df = spark.table(f"{params.silver_catalog}.information_schema.referential_constraints")

constraints_df = build_schema_constraints_df(
    columns_df, key_column_usage_df, table_constraints_df, referential_constraints_df, params.target_sample_fact
)
view(constraints_df.count(), params.execution_mode)
view(constraints_df, params.execution_mode)

In [ ]:
sample_source1_raw_df = spark.table(f"{params.bronze_catalog}.{params.bronze_schema}.source1_table")
sample_source2_raw_df = spark.table(f"{params.bronze_catalog}.{params.bronze_schema}.source2_table")
sample_source3_raw_df = spark.table(f"{params.bronze_catalog}.{params.bronze_schema}.source3_table")
parent1_df = spark.table(f"{params.silver_catalog}.{params.silver_schema}.parent1_dim")
parent2_df = spark.table(f"{params.silver_catalog}.{params.silver_schema}.parent2_dim")

sample_source1_clean_df = sample_source1_raw_df.select("col1", "col2", "col3", "col4", "col5")
sample_source2_clean_df = sample_source2_raw_df.select("col6", "col7", "col8")
sample_source3_clean_df = sample_source3_raw_df.select("col9", "col10", "col11")

reference_df = (
    sample_source1_clean_df.join(sample_source2_clean_df, F.col("col1") == F.col("col6"), "inner")
    .join(sample_source3_clean_df, F.col("col2") == F.col("col9"), "inner")
    .join(
        parent1_df,
        (F.col("parent1_num") == F.col("col12")) & (F.col("col2") == params.source_cd),
        "inner",
    )
    .select(
        F.concat_ws(
            "|",
            F.col("col3"),
            F.col("col4"),
            F.col("col10"),
            F.col("col7"),
            F.col("col8"),
        ).alias("sample_ck")
    )
)
parity_reference_col = "sample_ck"
parity_source_col = "sample_ck"
parent_dfs = {"parent1": parent1_df, "parent2": parent2_df}
business_rules = {
    "outliers": [{"column": "col5", "max": 10000}],
    "categories": [
        {
            "column": "col13",
            "allowed_values": ["value1", "value2", "value3", "value4"],
        }
    ],
}

clean_df, quarantine_df = data_quality_engine(
    source_df=sample_fact_df,
    target_table_name=params.target_sample_fact,
    constraints_df=constraints_df,
    reference_df=reference_df,
    parity_source_col=parity_source_col,
    parity_reference_col=parity_reference_col,
    parent_dfs=parent_dfs,
    business_rules=business_rules,
    mode="enforce",
)

In [ ]:
source = quarantine_df
target = f"{params.silver_catalog}.{params.silver_schema}.{params.target_sample_fact_quarantine}"
merge_key = "sample_ck"
write_mode = "append"
scope = "1=0"

scoped_merge_engine(spark, source, target, merge_key, write_mode, scope)

In [ ]:
source = sample_fact_df
target = f"{params.silver_catalog}.{params.silver_schema}.{params.target_sample_fact}"
merge_key = "sample_ck"
write_mode = "reload"
scope = f"t.sample_record_source_cd = ('{params.source_cd}')"

scoped_merge_engine(spark, source, target, merge_key, write_mode, scope)